# IEEE 57 + IEEE 14 connected by two HVDC grids

Two non-synchronous AC grids, the IEEE 57-bus and the IEEE 14-bus system, exchange power through two meshed multi-terminal HVDC grids. This is the multi-terminal HVDC case of the paper (section IV-B, Fig. 6), with the complete data of both IEEE systems (transformer taps, shunts, generator dispatch).

| | |
|---|---|
| AC buses | 72 (83 lines, 18 transformers, 10 generators) |
| DC buses | 10, in 2 DC grids |
| Converters | 8: 3 × VdcQ (holds its DC voltage, fixed Q), 5 × PQ (fixed P and Q) |
| Case file | `validation/cases/ieee57_14_hvdc.yaml` |

## What this notebook does

1. **Load** the case file into a pandapower net and classify the nodes. `LOSSY = True` adds the converter
   losses a + b·|I| + c·|I|² (coefficients of the paper, section IV-B).
2. **Build Y**: the AC admittance matrix (lines, transformer taps and shifts, shunts) and the DC conductance
   matrix.
3. **Solve** the AC grids, the DC grids and the converters together in one Newton–Raphson, and print the
   iterations, the largest mismatch and the run time.
4. **Results**: per node the voltage E = E_re + j·E_im (p.u.) and the injected power P, Q (p.u., positive into
   the grid); per converter its AC and DC power, losses and current.

## Try it

- Change a converter in the `converter` table of the case file, for example its `mode`, `p_mw`, `q_mvar` or
  `vdc_pu`, then *Run All*. Modes: PQ, VdcQ, PV, VdcV, GF (grid-forming) and droop; the columns each mode uses are
  listed at the top of the case file.
- Set `LOSSY = True` in the first cell to include converter losses.

## Grid topology

![Grid topology](figures/ieee57_14_hvdc.png)

The converters in the case file are named `IC <AC bus>-<DC bus>`.

## Running this notebook

Use the kernel **Python 3.12 (unified-acdc-powerflow)** (VS Code: *Select Kernel* → *Jupyter Kernel…*). To set it
up once, in the repository folder:

```bash
python3.12 -m venv .venv
.venv/bin/pip install -e ".[dev]"
.venv/bin/python -m ipykernel install --user --name unified-acdc-powerflow --display-name "Python 3.12 (unified-acdc-powerflow)"
```


In [1]:
import logging, warnings
logging.getLogger("pandapower").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_rows", 2000)
from unified_acdc_powerflow import build_model, build_ybus, solve
from unified_acdc_powerflow.cases import load_case

# 1) load the case and classify the nodes
CASE = "ieee57_14_hvdc"
LOSSY = False          # True: converter losses a + b|I| + c|I|^2 (coefficients of the paper)
net = load_case(CASE, lossy=LOSSY)   # pandapower net from validation/cases/ieee57_14_hvdc.yaml
model = build_model(net)
print(f"{model.n_ac} AC nodes, {model.n_dc} DC nodes, {model.n_conv} converters")
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu", "p_dc_ref_mw", "droop_k"]]


72 AC nodes, 10 DC nodes, 8 converters


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu,p_dc_ref_mw,droop_k
0,IC 212-1,212,1,VdcQ,0.0,-5.0,1.0,0.0,0.0
1,IC 150-2,150,2,PQ,20.0,21.0,1.0,0.0,0.0
2,IC 151-3,151,3,PQ,10.0,-31.0,1.0,0.0,0.0
3,IC 158-4,158,4,VdcQ,0.0,11.0,1.0,0.0,0.0
4,IC 202-5,202,5,VdcQ,0.0,-15.0,1.0,0.0,0.0
5,IC 205-6,205,6,PQ,45.0,30.0,1.0,0.0,0.0
6,IC 115-7,115,7,PQ,15.0,-40.0,1.0,0.0,0.0
7,IC 117-8,117,8,PQ,15.0,50.0,1.0,0.0,0.0


In [2]:
# 2) build Y (AC: pandapower Ybus with taps and shunts; DC: conductance matrix G)
build_ybus(model)
print("Y (AC):", model.Y.shape, "nnz", model.Y.nnz)
print("G (DC):", model.G.shape, "nnz", model.G.nnz)


Y (AC): (72, 72) nnz 270
G (DC): (10, 10) nnz 30


In [3]:
# 3) Newton-Raphson
res = solve(model)
print(res.message, "| iterations:", res.iterations, "| max mismatch:", f"{res.max_mismatch:.2e}",
      "| time:", f"{res.solve_time_s * 1e3:.1f} ms")


converged | iterations: 5 | max mismatch: 5.66e-14 | time: 83.1 ms


In [4]:
# 4) results: per node E (real, imag) and injected P, Q in p.u.; then per converter
display(res.table())
res.converter_table()


,side,bus,name,type,E_re,E_im,P_pu,Q_pu
0,ac,101,101,slack,1.040000,0.000000,4.531132e-01,1.452091e+00
1,ac,102,102,PV,1.009998,0.002088,-3.000000e-02,-8.935667e-01
2,ac,103,103,PV,0.984972,-0.007474,-1.000000e-02,-1.514507e-01
3,ac,104,104,PQ,0.980409,-0.015175,-9.511346e-16,-8.695145e-15
4,ac,105,105,PQ,0.976393,-0.012569,-1.300000e-01,-4.000000e-02
5,ac,106,106,PV,0.979994,-0.003519,-7.500000e-01,-3.766226e-02
6,ac,107,107,PQ,0.982626,0.034653,-8.515291e-16,-7.020692e-15
7,ac,108,108,PV,0.999807,0.102036,3.000000e+00,4.130059e-01
8,ac,109,109,PV,0.979544,0.029899,-1.210000e+00,-1.940412e-01
9,ac,110,110,PQ,0.978089,0.043001,-5.000000e-02,-2.000000e-02


,vsc,name,mode,P_ac_pu,Q_ac_pu,P_dc_pu,loss_pu,I_pu
0,0,IC 212-1,VdcQ,-0.300602,-0.05,0.300602,0.0,0.298649
1,1,IC 150-2,PQ,0.200000,0.21,-0.200000,0.0,0.276926
2,2,IC 151-3,PQ,0.100000,-0.31,-0.100000,0.0,0.314100
3,3,IC 158-4,VdcQ,-0.294227,0.11,0.294227,0.0,0.310417
4,4,IC 202-5,VdcQ,-0.559807,-0.15,0.559807,0.0,0.554598
5,5,IC 205-6,PQ,0.450000,0.30,-0.450000,0.0,0.523604
6,6,IC 115-7,PQ,0.150000,-0.40,-0.150000,0.0,0.435832
7,7,IC 117-8,PQ,0.150000,0.50,-0.150000,0.0,0.494718
